# F3. Fit engine: where the memory goes

`estimate()` returns a `FitResult` with a breakdown (weights, KV cache, overhead), a verdict,
a bandwidth-bound speed estimate, a confidence and the `formula_id` that produced it.

In [ ]:
from rightsize.catalog import facts
from rightsize.fit import estimate, gguf_bpw, kv_cache_gb
from rightsize.hardware import get

fx = facts("Qwen/Qwen3-4B")
dev = get("RTX 4070 Ti SUPER")
r = estimate(fx, "Q4_K_M", dev, ctx=8192)
r

Real bits per weight, not nominal: this is why predicted file sizes match `llama-quantize` output.

In [ ]:
{q: gguf_bpw(q) for q in ["Q4_K", "Q4_K_M", "Q5_K_M", "Q6_K", "Q8_0", "IQ4_XS"]}

## Context length changes the answer

In [ ]:
for ctx in [2048, 8192, 32768, 131072]:
    r = estimate(fx, "Q4_K_M", dev, ctx=ctx)
    print(
        f"ctx {ctx:6d}: kv {r.breakdown['kv_cache']:.2f} GB  total {r.vram_gb:.2f} GB  {r.verdict.value}"
    )

## Golden check: Llama 3.1 70B KV cache at 128K context is about 43 GB

In [ ]:
from rightsize.types import Family, ModelFacts, ModelRef

llama70 = ModelFacts(
    ref=ModelRef(repo="meta-llama/Llama-3.1-70B"),
    family=Family.llm,
    params_total=70_553_706_496,
    num_layers=80,
    num_kv_heads=8,
    head_dim=128,
)
kv_cache_gb(llama70, 131072)